In [ ]:
"""
=============================================================================
QUANTUM SVR — Projected Quantum Kernel (PQK)
Previsão de Zeros da Função Zeta de Riemann
=============================================================================

Referência principal:
  Huang et al. (2021) "Power of data in quantum machine learning"
  Nature Communications 12, 2631

Estratégia para SUPERAR o SVR clássico:
─────────────────────────────────────────────────────────────────────────────
  O SVR clássico usa RBF kernel sobre 10 features originais.
  
  O PQK expande 10 → ~200 features via circuito quântico ZZFeatureMap,
  computando valores esperados de observáveis de Pauli locais:

    φ(x) = [⟨X₁⟩, ⟨Y₁⟩, ⟨Z₁Z₂⟩, ⟨Z₁Z₃⟩, ..., ⟨ZᵢZⱼ⟩]
                                     ↑ 45 pares com d=10

  Depois aplica RBF sobre φ(x). Isso é garantidamente mais expressivo
  que RBF sobre x diretamente (o espaço de hipóteses contém o clássico).

Circuito por camada (2 reps):
  1. H⊗ⁿ                          → superposição uniforme
  2. Rz(2·xᵢ) ∀i                  → codificação single-qubit
  3. ZZ(i,j) = Rz(2·(π-xᵢ)(π-xⱼ)) → emaranhamento para todos os pares

Valores esperados (forma fechada):
  Estado após H·Rz(2xᵢ):   cos(2xᵢ)|0⟩ + i·sin(2xᵢ)|1⟩  (base X girada)
  ⟨Xᵢ⟩  = cos(2xᵢ) × ∏_{j≠i} cos(2(π-xᵢ)(π-xⱼ))
  ⟨Yᵢ⟩  = sin(2xᵢ) × ∏_{j≠i} cos(2(π-xᵢ)(π-xⱼ))
  ⟨ZᵢZⱼ⟩ = sin(2xᵢ)·sin(2xⱼ)·cos(2(π-xᵢ)(π-xⱼ))
=============================================================================
"""

import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, time
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAÇÕES
# ─────────────────────────────────────────────────────────────────────────────
DATA_PATH  = "../dataset/riemann_features.csv"
N_SAMPLES  = 2000
TEST_SPLIT = 0.80

FEATURES = [
    'z_co_gram_lag_2', 'z_gram',        'z_gram_lag_1', 'd_lag_13',
    'z_co_gram_lag_3', 'z_co_gram_lag_1','d_lag_14',    'd_lag_1',
    'z_gram_lag_2',    'd_lag_17'
]

RMSE_TARGET = 0.07
R2_TARGET   = 0.94

# ─────────────────────────────────────────────────────────────────────────────
# 1. CARREGAR DADOS
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("  QUANTUM SVR — PROJECTED QUANTUM KERNEL (PQK)")
print("=" * 65)

df = pd.read_csv(DATA_PATH)
X_raw = df[FEATURES].values[:N_SAMPLES]
y_all = df["distance"].values[:N_SAMPLES]

split    = int(TEST_SPLIT * N_SAMPLES)
X_tr_raw = X_raw[:split];  X_te_raw = X_raw[split:]
y_train  = y_all[:split];  y_test   = y_all[split:]

print(f"\nTreino: {len(y_train)}  |  Teste: {len(y_test)}  |  Features: {len(FEATURES)}")
print(f"Target — mean={y_all.mean():.4f}  std={y_all.std():.4f}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 2. ESCALAMENTO QUÂNTICO  →  [0, π]
# ─────────────────────────────────────────────────────────────────────────────
q_scaler = MinMaxScaler(feature_range=(0, np.pi))
X_tr_q   = q_scaler.fit_transform(X_tr_raw)
X_te_q   = q_scaler.transform(X_te_raw)

# ─────────────────────────────────────────────────────────────────────────────
# 3. MAPA DE FEATURES QUÂNTICO EXPLÍCITO  (ZZFeatureMap — PQK)
# ─────────────────────────────────────────────────────────────────────────────

def quantum_feature_map(X, n_reps=2):
    """
    Computa o mapa de features explícito do ZZFeatureMap (PQK).

    Para cada rep, calcula os valores esperados de Pauli:
      ⟨Xᵢ⟩, ⟨Yᵢ⟩  para cada qubit i
      ⟨ZᵢZⱼ⟩        para cada par (i,j)

    Retorna concatenação de features originais + todas as camadas.
    """
    n, d = X.shape
    pairs   = [(i, j) for i in range(d) for j in range(i+1, d)]
    n_pairs = len(pairs)   # C(d,2) = 45 para d=10

    x_cur    = X.copy()
    all_feats = []

    for rep in range(n_reps):
        zz_phase = np.pi - x_cur    # (n, d)

        # ── Fator de produto para ⟨Xᵢ⟩ e ⟨Yᵢ⟩ ──────────────────────────
        # log|cos| acumulado por qubit para evitar underflow
        log_cos = np.zeros((n, d))
        for (pi_idx, pj_idx) in pairs:
            angle   = 2.0 * zz_phase[:, pi_idx] * zz_phase[:, pj_idx]
            cos_val = np.cos(angle)
            safe    = np.clip(np.abs(cos_val), 1e-12, None)
            log_cos[:, pi_idx] += np.log(safe)
            log_cos[:, pj_idx] += np.log(safe)

        factor = np.exp(log_cos)           # ∏_{j≠i} |cos(ZZ_ij)|  (n, d)

        exp_X = np.cos(2.0 * x_cur) * factor    # (n, d)
        exp_Y = np.sin(2.0 * x_cur) * factor    # (n, d)

        # ── Correladores ZZ ──────────────────────────────────────────────
        exp_ZZ = np.zeros((n, n_pairs))
        for k, (pi_idx, pj_idx) in enumerate(pairs):
            angle = 2.0 * zz_phase[:, pi_idx] * zz_phase[:, pj_idx]
            exp_ZZ[:, k] = (
                np.sin(2.0 * x_cur[:, pi_idx])
                * np.sin(2.0 * x_cur[:, pj_idx])
                * np.cos(angle)
            )

        layer = np.hstack([exp_X, exp_Y, exp_ZZ])   # (n, 2d + n_pairs)
        all_feats.append(layer)

        # Próxima camada: reencoda os primeiros d valores esperados
        x_cur = MinMaxScaler(feature_range=(0, np.pi)).fit_transform(layer[:, :d])

    phi = np.hstack([X] + all_feats)   # (n, d + n_reps*(2d + n_pairs))
    return phi


print("── Computando mapa de features quântico ────────────────────")
t0 = time.time()
Phi_train = quantum_feature_map(X_tr_q, n_reps=2)
Phi_test  = quantum_feature_map(X_te_q, n_reps=2)
print(f"   Dim. original  : {X_tr_q.shape[1]}")
print(f"   Dim. quântica  : {Phi_train.shape[1]}")
print(f"   Tempo          : {time.time()-t0:.2f}s\n")

# ─────────────────────────────────────────────────────────────────────────────
# 4. NORMALIZAÇÃO
# ─────────────────────────────────────────────────────────────────────────────
std_scaler = StandardScaler()
Phi_train  = std_scaler.fit_transform(Phi_train)
Phi_test   = std_scaler.transform(Phi_test)

# ─────────────────────────────────────────────────────────────────────────────
# 5. TUNAGEM DE HIPERPARÂMETROS (TimeSeriesSplit)
# ─────────────────────────────────────────────────────────────────────────────
print("── Tunagem de hiperparâmetros (TimeSeriesSplit n=5) ─────────")

param_grid = {
    "svr__C"      : [50, 100, 200, 500, 1000],
    "svr__epsilon": [0.001, 0.005, 0.01, 0.02],
    "svr__gamma"  : ["scale", "auto", 0.01, 0.001],
}

tscv = TimeSeriesSplit(n_splits=5)

grid = GridSearchCV(
    Pipeline([("svr", SVR(kernel="rbf"))]),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    n_jobs=-1,
    verbose=0,
)
grid.fit(Phi_train, y_train)

best         = grid.best_params_
best_cv_rmse = -grid.best_score_
print(f"   Melhores params : {best}")
print(f"   CV RMSE         : {best_cv_rmse:.6f}\n")

model        = grid.best_estimator_
y_pred_train = model.predict(Phi_train)
y_pred_test  = model.predict(Phi_test)

# ─────────────────────────────────────────────────────────────────────────────
# 6. SVR CLÁSSICO (comparação direta, mesmo CV)
# ─────────────────────────────────────────────────────────────────────────────
print("── SVR Clássico (baseline) ──────────────────────────────────")

std_c      = StandardScaler()
X_tr_c     = std_c.fit_transform(X_tr_raw)
X_te_c     = std_c.transform(X_te_raw)

classic_grid = GridSearchCV(
    Pipeline([("svr", SVR(kernel="rbf"))]),
    {"svr__C": [50, 100, 500, 1000],
     "svr__epsilon": [0.001, 0.01, 0.02],
     "svr__gamma": ["scale", "auto", 0.01]},
    scoring="neg_root_mean_squared_error",
    cv=tscv, n_jobs=-1, verbose=0
)
classic_grid.fit(X_tr_c, y_train)
y_pred_classic = classic_grid.best_estimator_.predict(X_te_c)
classic_rmse   = np.sqrt(mean_squared_error(y_test, y_pred_classic))
classic_r2     = r2_score(y_test, y_pred_classic)
print(f"   CV RMSE clássico: {-classic_grid.best_score_:.6f}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 7. RESULTADOS
# ─────────────────────────────────────────────────────────────────────────────
test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2    = r2_score(y_test, y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_r2   = r2_score(y_train, y_pred_train)

print("=" * 65)
print("  RESULTADOS FINAIS")
print("=" * 65)
print(f"  SVR Clássico  → RMSE={classic_rmse:.6f}  R²={classic_r2:.6f}")
print(f"  SVR Quântico  → RMSE={test_rmse:.6f}  R²={test_r2:.6f}")
print(f"  Melhoria RMSE : {classic_rmse - test_rmse:+.6f}  ({(classic_rmse-test_rmse)/classic_rmse*100:+.1f}%)")
print(f"  Melhoria R²   : {test_r2 - classic_r2:+.6f}")
print("=" * 65)
ok_rmse = test_rmse < RMSE_TARGET
ok_r2   = test_r2   > R2_TARGET
print(f"\n  Alvo: RMSE < {RMSE_TARGET}  |  R² > {R2_TARGET}")
if ok_rmse and ok_r2:
    print("  ✅  AMBOS OS OBJETIVOS ATINGIDOS!")
else:
    print(f"  {'✅' if ok_rmse else '❌'}  RMSE={test_rmse:.5f}")
    print(f"  {'✅' if ok_r2   else '❌'}  R²  ={test_r2:.5f}")

# ─────────────────────────────────────────────────────────────────────────────
# 8. GRÁFICOS
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Quantum SVR (PQK) — Zeros da Função Zeta de Riemann",
             fontsize=13, fontweight='bold')

mn, mx = y_test.min(), y_test.max()

ax = axes[0, 0]
ax.scatter(y_test, y_pred_test, alpha=0.4, s=18, color='royalblue')
ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
ax.set_title(f"Quântico: Predito vs Real\nRMSE={test_rmse:.5f}  R²={test_r2:.5f}")
ax.set_xlabel("Real"); ax.set_ylabel("Predito"); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.scatter(y_test, y_pred_classic, alpha=0.4, s=18, color='tomato')
ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
ax.set_title(f"Clássico: Predito vs Real\nRMSE={classic_rmse:.5f}  R²={classic_r2:.5f}")
ax.set_xlabel("Real"); ax.set_ylabel("Predito"); ax.grid(alpha=0.3)

ax = axes[0, 2]
labels = ['RMSE (↓)', 'R² (↑)']
xp     = np.arange(2)
ax.bar(xp - 0.2, [test_rmse,   test_r2],   0.35, label='Quântico', color='royalblue', alpha=0.85)
ax.bar(xp + 0.2, [classic_rmse, classic_r2], 0.35, label='Clássico', color='tomato',   alpha=0.85)
ax.axhline(RMSE_TARGET, color='red',   ls=':', lw=1.5, label=f'Alvo RMSE={RMSE_TARGET}')
ax.axhline(R2_TARGET,   color='green', ls=':', lw=1.5, label=f'Alvo R²={R2_TARGET}')
ax.set_xticks(xp); ax.set_xticklabels(labels)
ax.set_title("Comparação: Quântico vs Clássico")
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

ax = axes[1, 0]
n_show = min(300, len(y_test))
idx    = np.arange(n_show)
ax.plot(idx, y_test[:n_show],         label='Real',     color='steelblue', lw=1.2)
ax.plot(idx, y_pred_test[:n_show],    label='Quântico', color='royalblue', lw=1, ls='--')
ax.plot(idx, y_pred_classic[:n_show], label='Clássico', color='tomato',    lw=1, ls=':')
ax.set_title("Série Temporal — Conjunto de Teste")
ax.set_xlabel("Índice"); ax.set_ylabel("Distance"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 1]
res_q = y_test - y_pred_test
res_c = y_test - y_pred_classic
ax.hist(res_q, bins=40, alpha=0.65, color='royalblue', label=f'Quântico (std={res_q.std():.4f})')
ax.hist(res_c, bins=40, alpha=0.65, color='tomato',    label=f'Clássico (std={res_c.std():.4f})')
ax.axvline(0, color='black', lw=2, ls='--')
ax.set_title("Distribuição dos Resíduos")
ax.set_xlabel("Resíduo"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 2]
feat_var = np.var(Phi_train, axis=0)
top_k    = 20
top_idx  = np.argsort(feat_var)[-top_k:][::-1]
ax.barh(range(top_k), feat_var[top_idx], color='mediumpurple', alpha=0.85)
ax.set_yticks(range(top_k))
ax.set_yticklabels([f"φ_{i}" for i in top_idx], fontsize=8)
ax.set_title(f"Top-{top_k} Features Quânticas (Variância)")
ax.set_xlabel("Variância"); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/quantum_svr_results.png", dpi=150, bbox_inches='tight')
print("\nGráfico salvo em: outputs/quantum_svr_results.png")

print("\n" + "─" * 65)
print("  RESUMO")
print("─" * 65)
print(f"  Abordagem     : Projected Quantum Kernel (PQK)")
print(f"  Circuito      : ZZFeatureMap  reps=2  d={len(FEATURES)} qubits")
print(f"  Dim. original : {len(FEATURES)}")
print(f"  Dim. quântica : {Phi_train.shape[1]}")
print(f"  Observáveis   : ⟨Xᵢ⟩, ⟨Yᵢ⟩ ∀i  +  ⟨ZᵢZⱼ⟩ ∀i<j  × 2 reps")
print(f"  Kernel SVR    : RBF sobre φ(x)")
print(f"  C={best['svr__C']}  ε={best['svr__epsilon']}  γ={best['svr__gamma']}")
print("─" * 65)